# Week 6: Data Visualization Mastery - Interactive Dashboard
Advanced Seaborn + Plotly visualization project

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
sns.set_palette('husl')
%matplotlib inline

## Load & Explore Data

In [ ]:
# Load data
sales = pd.read_csv('data/sales_data.csv')
customers = pd.read_csv('data/customer_churn.csv')

print(f"Sales shape: {sales.shape}")
print(f"Customers shape: {customers.shape}")
print(f"\nSales columns: {sales.columns.tolist()}")
print(f"Customer columns: {customers.columns.tolist()}")

## Prepare Data

In [ ]:
# Merge datasets
sales['Date'] = pd.to_datetime(sales['Date'])
sales['Month'] = sales['Date'].dt.to_period('M').astype(str)

merged = pd.merge(sales, customers,
                  left_on='Customer_ID',
                  right_on='CustomerID',
                  how='left')

print(f"Merged shape: {merged.shape}")
print(merged.head())

## Seaborn Visualizations

In [ ]:
# Box Plot
plt.figure(figsize=(12, 6))
sns.boxplot(data=merged, x='Product', y='Price', palette='Set2')
plt.title('Price Distribution by Product', fontsize=14, fontweight='bold')
plt.ylabel('Price ($)')
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 8))
numeric = merged.select_dtypes(include=[np.number]).corr()
sns.heatmap(numeric, annot=True, fmt='.2f', cmap='RdYlGn', center=0, square=True)
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.show()

## Plotly Interactive Visualizations

In [ ]:
# Interactive Dashboard
product_sales = merged.groupby('Product')['Total_Sales'].sum().sort_values(ascending=False)
region_sales = merged.groupby('Region')['Total_Sales'].sum()
monthly = merged.groupby('Month')['Total_Sales'].sum().reset_index().sort_values('Month')
churn = merged.groupby('Churn')['CustomerID'].count()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Sales by Product', 'Sales by Region', 'Monthly Trends', 'Churn')
)

fig.add_trace(go.Bar(x=product_sales.index, y=product_sales.values), row=1, col=1)
fig.add_trace(go.Pie(labels=region_sales.index, values=region_sales.values), row=1, col=2)
fig.add_trace(go.Scatter(x=monthly['Month'], y=monthly['Total_Sales'], mode='lines+markers'), row=2, col=1)
fig.add_trace(go.Bar(x=['Retained', 'Churned'], y=[churn.get(0, 0), churn.get(1, 0)]), row=2, col=2)

fig.update_layout(height=800, showlegend=True, title_text='Interactive Dashboard')
fig.show()